# Lesson 02 — CART Regression Loss and Split Selection

## Objective: replace a guessed threshold with a data-driven choice

In Lesson 01, we manually chose `MedInc = 4.0` to split observations into two leaves. This lesson replaces that human guess with the core CART split-selection procedure: generate every meaningful candidate threshold, evaluate the predictions produced by each split, and choose the threshold with the smallest total squared error.

The midpoint thresholds are only the finite set of split proposals that CART needs to test. For each proposal, we will:

1. divide the rows into left (`x ≤ threshold`) and right (`x > threshold`) children;
2. use each child's mean target as its leaf prediction;
3. calculate the squared-error loss in each child;
4. add the child losses to obtain the total split loss; and
5. select the threshold with the lowest total loss.

By the end of the lesson, you should be able to calculate and implement the best split for one feature at one node. We will not yet recurse, grow a complete tree, or consider multiple features.

## Reuse the deterministic California Housing slice

We reuse the same 12 real rows from Lesson 01 so that the new split calculation stays connected to familiar data. The rows are sorted by `MedInc`, making every possible boundary visible between neighboring feature values.

For row $i$:

- $x_i$ is its `MedInc` feature value, measured in units of $10,000; and
- $y_i$ is its observed `MedHouseVal` target, measured in units of $100,000.

The arrays `x` and `y` preserve this displayed order.

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
housing_rows = housing.frame.loc[:, ["MedInc", "MedHouseVal"]]

sorted_rows = (
    housing_rows.reset_index(names="source_row")
    .sort_values(by=["MedInc", "source_row"])
    .reset_index(drop=True)
)

rank_positions = np.linspace(
    0,
    len(sorted_rows) - 1,
    num=12,
    dtype=int,
)

housing_slice = sorted_rows.iloc[rank_positions].reset_index(drop=True)

x = housing_slice["MedInc"].to_numpy()
y = housing_slice["MedHouseVal"].to_numpy()

housing_slice

,source_row,MedInc,MedHouseVal
0,73,0.4999,0.675
1,241,1.8472,1.370
2,3789,2.2756,2.214
3,19601,2.6406,1.297
4,5841,3.0179,2.725
5,5606,3.3427,1.542
6,19373,3.7031,1.733
7,6743,4.1136,2.799
8,16665,4.6071,2.399
9,8554,5.2500,2.472


## Turn one promising gap into a candidate split

By eye, the gap between rows 9 and 10 looks promising because it separates the targets `4.500` and `4.000` from the mostly lower targets before them. We represent that gap with its midpoint threshold:

$$
t = \frac{x_9 + x_{10}}{2} = 5.7723.
$$

Rows with $x_i \le t$ enter the left child; rows with $x_i > t$ enter the right child. Each child predicts the mean of the targets assigned to it:

$$
\hat{y}_L = \operatorname{mean}(y_i : x_i \le t), \qquad \hat{y}_R = \operatorname{mean}(y_i : x_i > t).
$$

These map to `threshold`, `left_mask`, `right_mask`, `left_prediction`, and `right_prediction` in the code below.

In [2]:
threshold = (x[9] + x[10]) / 2

left_mask = x <= threshold
right_mask = x > threshold

left_prediction = y[left_mask].mean()
right_prediction = y[right_mask].mean()

threshold, left_prediction, right_prediction

(np.float64(5.7722999999999995),
 np.float64(1.9226000000000003),
 np.float64(4.25))

### Check the partition

This candidate should send 10 rows left and 2 rows right. The assertions also verify the essential partition rule: every row reaches one child, and no row reaches both children. Passing assertions produce no output.

In [3]:
assert left_mask.sum() == 10
assert right_mask.sum() == 2
assert not (left_mask & right_mask).any()
assert (left_mask | right_mask).all()

## Measure how well this candidate predicts

A **residual** is the difference between an observed target and the prediction it receives. For example, a left-row residual is $y_i - \hat{y}_L$. Squaring prevents negative and positive misses from cancelling.

**RSS stands for Residual Sum of Squares.** A residual is one target-minus-prediction difference; RSS squares those residuals and adds them. In the sums below, $i : x_i \le t$ means ‘each row index $i$ whose feature value $x_i$ is at most the threshold $t$.’ Similarly, $i : x_i > t$ means the rows assigned to the right child. The left and right leaf losses are:

$$
\operatorname{RSS}_L = \sum_{i : x_i \le t}(y_i - \hat{y}_L)^2, \qquad \operatorname{RSS}_R = \sum_{i : x_i > t}(y_i - \hat{y}_R)^2.
$$

The candidate needs one overall score, so we add the child losses:

$$
L_{\mathrm{split}}(t) = \operatorname{RSS}_L + \operatorname{RSS}_R.
$$

In code, these quantities are `left_residuals`, `right_residuals`, `left_loss`, `right_loss`, and `split_loss`. A lower `split_loss` means the two leaf predictions fit their assigned targets more closely.

In [4]:
left_residuals = y[left_mask] - left_prediction
right_residuals = y[right_mask] - right_prediction

# Sum the squared left residuals.
left_loss = (left_residuals**2).sum()

# Sum the squared right residuals.
right_loss = (right_residuals**2).sum()

# Combine both child losses into one candidate score.
split_loss = left_loss + right_loss

left_loss, right_loss, split_loss

(np.float64(4.459686400000001),
 np.float64(0.125),
 np.float64(4.584686400000001))

### Check the candidate score

For this visible split, the left RSS is approximately `4.4597`, the right RSS is `0.125`, and their sum is approximately `4.5847`. `np.isclose` checks these decimal results with a small floating-point tolerance instead of requiring exact binary equality. This score evaluates only the candidate $t = 5.7723$; we have not yet proved that it is the best candidate.

In [5]:
assert np.isclose(left_loss, 4.4596864)
assert np.isclose(right_loss, 0.125)
assert np.isclose(split_loss, left_loss + right_loss)